# 01 — Data Loading

Builds the modelling spine from a raw Letterboxd export.

**Input:** a Letterboxd export directory (path set via `LETTERBOXD_EXPORT_DIR` in `.env`)
**Output:** `data/interim/viewings.csv` — one row per rated viewing event

The export splits the same viewing history across several files with different
columns and row counts. This notebook establishes how they relate, picks a spine,
joins the rest onto it, and cleans the review text ready for NLP later.


## 1. Load the export

The export path is read from `.env` rather than hardcoded, so the notebook runs on
any Letterboxd export rather than only this one. Required files fail loudly if
absent; optional files are loaded when present, since not every account has a
watchlist or watched list.


In [ ]:
import html
import os
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

REQUIRED_FILES = ["diary.csv", "reviews.csv", "ratings.csv"]
OPTIONAL_FILES = ["watchlist.csv", "watched.csv"]

def load_export(export_dir=None):
    """Load a Letterboxd export directory into a dict of dataframes."""
    export_dir = Path(export_dir or os.getenv("LETTERBOXD_EXPORT_DIR", ""))

    if not export_dir.is_dir():
        raise FileNotFoundError(
            f"Export directory not found: {export_dir!s}\n"
            "Set LETTERBOXD_EXPORT_DIR in your .env, or pass export_dir explicitly."
        )

    missing = [f for f in REQUIRED_FILES if not (export_dir / f).exists()]
    if missing:
        raise FileNotFoundError(f"Export is missing required files: {missing}")

    data = {f.replace(".csv", ""): pd.read_csv(export_dir / f) for f in REQUIRED_FILES}

    for f in OPTIONAL_FILES:
        if (export_dir / f).exists():
            data[f.replace(".csv", "")] = pd.read_csv(export_dir / f)

    return data

export = load_export()

for name, df in export.items():
    print(f"{name:10s} {str(df.shape):12s} {list(df.columns)}")

## 2. Establish how the files relate

Three questions decide the shape of everything downstream:

- Can `reviews.csv` be joined onto `diary.csv` cleanly?
- Is every rated film present in the diary?
- How many films sit outside the diary entirely?

`film_key` is title-plus-year rather than the Letterboxd URI, because a diary URI
identifies a *viewing* while a ratings URI identifies a *film* — only title and year
work across all files.


In [ ]:
diary     = export["diary"]
reviews   = export["reviews"]
ratings   = export["ratings"]
watched   = export["watched"]

def film_key(df, title_col="Name", year_col="Year"):
    return df[title_col].str.strip() + " (" + df[year_col].astype("Int64").astype(str) + ")"

print("diary   : rows", len(diary),   "| unique URIs", diary["Letterboxd URI"].nunique(),
      "| unique films", film_key(diary).nunique())
print("reviews : rows", len(reviews), "| unique URIs", reviews["Letterboxd URI"].nunique())
print("ratings : rows", len(ratings), "| unique films", film_key(ratings).nunique())
print("watched : rows", len(watched), "| unique films", film_key(watched).nunique())
print()
print("diary URIs == reviews URIs :", set(diary["Letterboxd URI"]) == set(reviews["Letterboxd URI"]))
print("ratings films not in diary :", len(set(film_key(ratings)) - set(film_key(diary))))
print("diary films not in ratings :", len(set(film_key(diary)) - set(film_key(ratings))))
print("watched films not in diary :", len(set(film_key(watched)) - set(film_key(diary))))

**Findings.** Diary and reviews are 1:1 on URI, and all 1,176 rated films appear in
the diary, so **diary is the spine and reviews joins onto it**.

`reviews.csv` carries every diary column plus `Review`, so it could serve as the spine
for *this* export — but only because this account reviews everything. Typical review
coverage is 10–20%, and a pipeline built on `reviews.csv` would silently discard most
of another user's films.

`watched.csv` holds 890 films absent from the diary: pre-Letterboxd viewing, bulk
marked when the account was created. No rating and no reliable date, so they cannot
be training rows. Parked as possible exposure-count features (further work).


## 3. Build the spine

`validate="one_to_one"` asserts the join relationship verified above, so another
user's malformed export fails loudly rather than silently duplicating rows.

`Tags` is dropped: user-defined free text, inconsistent between accounts by
construction, not modellable.


In [ ]:
spine = diary.merge(
    reviews[["Letterboxd URI", "Review"]],
    on="Letterboxd URI",
    how="left",
    validate="one_to_one",
)

spine = spine.rename(columns={
    "Name":           "film_title",
    "Year":           "film_year",
    "Letterboxd URI": "entry_uri",
    "Rating":         "rating",
    "Date":           "logged_date",
    "Watched Date":   "watched_date",
    "Review":         "review_raw",
    "Rewatch":        "is_rewatch",
})

spine["logged_date"]  = pd.to_datetime(spine["logged_date"])
spine["watched_date"] = pd.to_datetime(spine["watched_date"])
spine["is_rewatch"]   = spine["is_rewatch"].eq("Yes")
spine["film_key"] = film_key(spine, "film_title", "film_year")

spine = spine.drop(columns=["Tags"])

print(spine.shape)
print(spine[["film_title", "film_year", "watched_date", "rating", "is_rewatch"]].head())
print()
print(spine.isna().sum())

## 4. Drop unrated entries

No rating means no target. The check runs before the drop so that on any export it is
visible what is being discarded — a bug that removes half a user's data should not be
silent.


In [ ]:
unrated = spine[spine["rating"].isna()]

print(f"unrated rows: {len(unrated)} ({len(unrated)/len(spine)*100:.1f}%)")
print(f"review length — unrated median: {unrated['review_raw'].str.split().str.len().median():.0f} words")
print(f"review length — rated median:   {spine[spine['rating'].notna()]['review_raw'].str.split().str.len().median():.0f} words")
print()
print(unrated[["film_title", "film_year"]].head(15).to_string(index=False))

Unrated entries are dropped as they have no target; in this export they are almost entirely short films, which would also distort the runtime and genre distributions


In [ ]:
before = len(spine)
spine = spine[spine["rating"].notna()].copy()
print(f"dropped {before - len(spine)} unrated rows -> {len(spine)} remain")

## 5. Clean review text

Letterboxd stores reviews as HTML fragments — 64% of these contain markup. Four steps,
each mattering for the TF-IDF work later:

1. `html.unescape` **first**, so `&amp;` becomes `&` rather than surviving as a token
2. Tags replaced with a **space**, not an empty string — `<i>Dune </i>suffers` must not
   collapse into one token
3. Smart quotes normalised, so `don't` and `don’t` do not become separate features
4. Whitespace collapsed, handling the `\r\n` line breaks in the raw text


In [ ]:
TAG_RE   = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")
SMART = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"',
                       "\u201d": '"', "\u2013": "-", "\u2014": "-", "\u2026": "..."})

def clean_review(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)      # &amp; -> &
    text = TAG_RE.sub(" ", text)    # strip <i>, <b>, <a href=...>
    text = text.translate(SMART)    # curly quotes -> straight
    return SPACE_RE.sub(" ", text).strip()

spine["review_clean"] = spine["review_raw"].apply(clean_review)
spine["review_words"] = spine["review_clean"].str.split().str.len()

print(spine["review_words"].describe([.1, .5, .9]).round(1))
print()
print(spine["review_clean"].iloc[0][:300])

Verification that the cleaning did what was intended. A handful of surviving `<`
characters would be genuine text ("less than") rather than markup, and should not be
forced to zero.


In [ ]:
print(spine["review_clean"].str.contains("<").sum(), "reviews still contain '<'")
print(spine["review_clean"].str.contains("’").sum(), "still contain curly apostrophes")
print()
print(spine["review_clean"].iloc[0][:300])

**Note for later.** Review length runs from 1 to 861 words, median 77. That spread
matters for TF-IDF, and `review_words` is a candidate feature in its own right —
revisit at Model 4.


## 6. Chronological ordering

Everything downstream depends on this. Expanding-window history features iterate over
`viewing_index`, and the train/test split is chronological.

Sorted on `watched_date` with `logged_date` as tiebreaker: several films are logged on
the same day, and without a deterministic secondary sort the history features would
depend on CSV row order.


In [ ]:
spine = spine.sort_values(["watched_date", "logged_date"]).reset_index(drop=True)

spine["viewing_index"]   = range(1, len(spine) + 1)
spine["film_viewing_seq"] = spine.groupby("film_key").cumcount() + 1
spine["film_decade"]      = (spine["film_year"] // 10) * 10
spine["film_age_at_watch"] = spine["watched_date"].dt.year - spine["film_year"]

print(f"{spine['watched_date'].min().date()} -> {spine['watched_date'].max().date()}")
print()
print(spine["watched_date"].dt.year.value_counts().sort_index())
print()
print(f"films watched more than once: {(spine['film_viewing_seq'] > 1).sum()}")

## 7. Save

Relative path, unlike the export directory: the export is machine-specific and lives in
`.env`, while intermediate outputs belong inside the project. `data/` is gitignored —
the export is personal data.


In [ ]:
OUT = Path("data/interim")
OUT.mkdir(parents=True, exist_ok=True)

spine.to_csv(OUT / "viewings.csv", index=False)
print(f"saved {len(spine)} rows, {spine.shape[1]} columns -> {OUT / 'viewings.csv'}")
print(list(spine.columns))

---

## Summary

| | |
|---|---|
| Rated viewing events | 1,194 |
| Unique films | 1,176 |
| Date span | 25 Dec 2021 – 31 Aug 2026 |
| Review coverage | 100% (atypical — see §2) |
| Rewatches with a prior entry | 18 |

Diary as spine, reviews joined on URI, tags and unrated entries dropped, review HTML
cleaned, chronological order established.

**Deferred:** validation for malformed exports — missing `Watched Date`, retrospective
import dumps, minimum-data gates. Written once the pipeline shows what it needs.

**Next:** `02_tmdb_enrichment.ipynb` reads `viewings.csv` and matches films to TMDB.
